# 02. Pose Baseline

## Goal

U3에서는 Pose 모델을 실행하기 전에 `b00_sn_f_01` 원본 영상만 보고 다섯 이벤트 프레임을 기록합니다. 검증된 이벤트를 모든 프레임의 다섯 동작 구간으로 확장해 이후 Pose 분석의 비교 기준을 만듭니다.

## 현재 범위

이 노트북의 U3 구역에는 YOLO Pose 결과가 없습니다. 모델 결과를 먼저 보면 수동 기준이 영향을 받을 수 있으므로, 이벤트 5개를 확정하고 저장한 다음 U5에서 같은 노트북에 Pose 구역을 추가합니다.

## Setup

프로젝트 루트 또는 `notebooks/` 디렉터리에서 실행할 수 있습니다. 저장과 검증 로직은 `phases.py`, 프레임 읽기는 `video.py`를 사용합니다.

In [ ]:
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.config import ARTIFACT_ROOT, B00_VIDEO_ID
from snatch_analysis.phases import EVENT_NAMES, prepare_phase_annotations
from snatch_analysis.video import read_video_frame, read_video_properties


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## Steps

### 1. 경로와 저장 설정

`OVERWRITE_PHASE_EVENTS=False`가 기본값입니다. 이미 B00 수동 이벤트가 있으면 사용자가 편집한 원본을 그대로 재사용합니다. 다섯 프레임을 다시 확정했을 때만 `True`로 바꾸세요.

In [ ]:
VIDEO_ID = B00_VIDEO_ID
MANIFEST_PATH = Path("data/metadata/videos.csv")
ANNOTATION_PATH = Path("data/annotations/phase_events.csv")
ARTIFACT_DIRECTORY = ARTIFACT_ROOT / VIDEO_ID
PREVIEW_FRAME_INDEX = 0
OVERWRITE_PHASE_EVENTS = False
ANNOTATOR = ""
NOTES = ""

### 2. B00 영상 기준 확인

U1 Manifest와 실제 영상 속성이 일치하는지 다시 확인합니다. 이 단계에서 확인한 `frame_count`가 이벤트 범위와 `phase_by_frame.csv` 행 수의 기준입니다.

In [ ]:
if VIDEO_ID != B00_VIDEO_ID:
    raise ValueError(f"U3는 {B00_VIDEO_ID}만 지원합니다: {VIDEO_ID}")
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Manifest가 없습니다: {MANIFEST_PATH}. 01_data_diagnosis.ipynb를 먼저 실행하세요."
    )

manifest = pd.read_csv(MANIFEST_PATH)
selected_rows = manifest.loc[manifest["video_id"] == VIDEO_ID]
if len(selected_rows) != 1:
    raise ValueError(
        f"Manifest에서 VIDEO_ID={VIDEO_ID!r} 행이 정확히 하나여야 합니다: "
        f"{len(selected_rows)}개"
    )

selected_video = selected_rows.iloc[0]
video_path = Path(selected_video["path"])
frame_count = int(selected_video["frame_count"])
video_properties = read_video_properties(video_path)
if video_properties["read_status"] != "ok":
    raise RuntimeError("B00 영상 기준 실패: read_status_ok")

video_checks = {
    "frame_count_matches": video_properties["frame_count"] == frame_count,
    "fps_matches": abs(
        float(video_properties["fps"]) - float(selected_video["fps"])
    ) < 1e-6,
    "size_matches": (
        video_properties["width"],
        video_properties["height"],
    )
    == (int(selected_video["width"]), int(selected_video["height"])),
}
failed_video_checks = [
    name for name, passed in video_checks.items() if not passed
]
if failed_video_checks:
    raise RuntimeError(
        f"B00 영상 기준 실패: {', '.join(failed_video_checks)}"
    )

pd.Series(
    {
        "video_id": VIDEO_ID,
        "FPS": video_properties["fps"],
        "frame_range": f"0 ~ {frame_count - 1}",
        "frame_count": frame_count,
        "decoded_size": (
            video_properties["height"],
            video_properties["width"],
            3,
        ),
        "contract": "PASS",
    },
    name="value",
).to_frame()

### 3. 원본 프레임을 보며 이벤트 탐색

`PREVIEW_FRAME_INDEX`를 바꾸고 이 셀만 다시 실행하면서 동작 전환점을 찾으세요. Pose 점이나 원판 bbox는 표시하지 않습니다. OpenCV BGR 프레임은 화면에 표시할 때만 RGB로 변환합니다.

In [ ]:
preview_frame = read_video_frame(video_path, PREVIEW_FRAME_INDEX)
preview_rgb = cv2.cvtColor(preview_frame, cv2.COLOR_BGR2RGB)
preview_time_sec = PREVIEW_FRAME_INDEX / float(video_properties["fps"])

plt.figure(figsize=(6, 8))
plt.imshow(preview_rgb)
plt.title(
    f"{VIDEO_ID} | frame {PREVIEW_FRAME_INDEX} | {preview_time_sec:.3f}s"
)
plt.axis("off")
plt.show()

### 4. 다섯 이벤트 프레임 입력

프레임을 충분히 확인한 뒤 `None`을 정수 frame index로 바꾸세요. 조건은 `bar_off_floor < knee_pass < max_extension < catch_stable < recovery_stable`이며 모두 `0..frame_count-1` 안에 있어야 합니다. 기존 원본을 재사용할 때는 기본값을 그대로 두어도 됩니다.

In [ ]:
EVENT_FRAMES = {
    "bar_off_floor": None,
    "knee_pass": None,
    "max_extension": None,
    "catch_stable": None,
    "recovery_stable": None,
}

### 5. 검증하고 로컬 CSV 저장

처음 저장할 때는 다섯 값이 모두 필요합니다. 원본 `data/annotations/phase_events.csv`는 기본적으로 재사용하며, 현재 실행에 검증된 B00 행과 모든 프레임 구간은 ignored `artifacts/b00_sn_f_01/`에 snapshot으로 저장합니다.

In [ ]:
phase_events, phase_by_frame, source_action = prepare_phase_annotations(
    video_id=VIDEO_ID,
    event_frames=EVENT_FRAMES,
    frame_count=frame_count,
    annotation_path=ANNOTATION_PATH,
    artifact_directory=ARTIFACT_DIRECTORY,
    annotator=ANNOTATOR,
    notes=NOTES,
    overwrite=OVERWRITE_PHASE_EVENTS,
)

In [ ]:
pd.Series(
    {
        "source_action": source_action,
        "event_rows": len(phase_events),
        "phase_rows": len(phase_by_frame),
        "source_path": ANNOTATION_PATH.as_posix(),
        "snapshot_path": (ARTIFACT_DIRECTORY / "phase_by_frame.csv").as_posix(),
        "manual_checkpoint": "PASS",
    },
    name="value",
).to_frame()

## Checks

### 6. 저장된 이벤트와 구간 수 확인

이벤트 5행과 모든 프레임의 구간이 생성됐는지 확인합니다. `recovery_complete=True`는 `recovery_stable` 프레임부터 표시됩니다.

In [ ]:
phase_events

In [ ]:
phase_order = [
    "Set-up",
    "First Pull",
    "Second Pull",
    "Drop Under & Catch",
    "Recovery",
]
phase_by_frame["phase"].value_counts().reindex(
    phase_order, fill_value=0
).rename("frame_count").to_frame()

In [ ]:
event_index_set = set(phase_events["frame_index"].astype(int))
boundary_indices = sorted(
    {0, frame_count - 1}
    | event_index_set
    | {index - 1 for index in event_index_set if index > 0}
)
phase_by_frame.loc[
    phase_by_frame["frame_index"].isin(boundary_indices)
].reset_index(drop=True)

## 관찰 기록

- `bar_off_floor`을 선택한 근거:
- `knee_pass`를 선택한 근거:
- `max_extension`을 선택한 근거:
- `catch_stable`을 선택한 근거:
- `recovery_stable`을 선택한 근거:
- 판단이 어려웠던 경계와 후보 프레임:

## Next Steps

다섯 이벤트를 직접 확인하고 `manual_checkpoint=PASS`가 나온 뒤에만 U5 Pose 결과를 확인합니다. U4에서는 모델과 독립적인 관절 각도 함수를 먼저 구현합니다.